# 04 — Train + Seed Evaluation (Stages 8–9)
**Does:** one skip-gram model per (period × seed) from validated shards, then a formal stability comparison across seeds with a prespecified production-seed rule. This replaces the old Notebook 07 (archived) with audit fixes.
**Reads:** frozen `period_definitions.csv`, `shards/tokenized/`, config v0.3.0. **Writes:** `models/word2vec/<corpus>/<group>/`, sidecar `.nfo.json`, checkpoints, `training_manifest.csv` (registry), refreshed `metadata/corpus_master.csv`, `production_seed.json`, seed diagnostics.
**Fixes vs 07:** registry todo logic rewritten (07 crashed on non-empty registries), tmp-dir fallback (07 crashed off-Colab), string version compare replaced.


In [ ]:
# Cell 1 — RUN PLAN (the only cell you must edit).
MODEL_FILTER_SUB = None   # e.g. 'AskAcademia' proving run; None = all sufficient periods
PERIOD_FILTER = None        # e.g. ['w2v__askacademia__2019q1']; None = all
SEED_FILTER = None          # e.g. [1047]; None = all config seed_candidates
MAX_MODELS = 2              # cap on (period x seed) trainings; None = uncapped production
DRY_RUN = False             # True = synthetic shards + throwaway dim-10 model in tmp only
print(f"sub={MODEL_FILTER_SUB} periods={PERIOD_FILTER} seeds={SEED_FILTER} cap={MAX_MODELS} dry={DRY_RUN}")


In [ ]:
# Bootstrap: locate the repo root and add it to sys.path so the shared 'src'
# package is importable regardless of the notebook's current working directory.
import os, sys
from pathlib import Path
def _has_root_marker(p):
    try:
        return p.is_dir() and (p / "config/project_config.yaml").is_file()
    except OSError:
        return False
def _find_root(start):
    for p in [start, *start.parents]:
        if _has_root_marker(p):
            return p
    for depth in (1, 2):
        for sub in start.glob("/".join(["*"] * depth)):
            if sub.is_dir() and _has_root_marker(sub):
                return sub
    return None
_repo_root = _find_root(Path.cwd().resolve())
if _repo_root is not None and str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
# Cell 2 — Setup: root, config, gensim, logger, training manifest, helpers.
import os, sys, csv, json, gzip, time, gc, hashlib, subprocess, datetime
from pathlib import Path
import yaml
from src.paths import get_project_root, resolve_tmp
from src.storage import atomic_write_bytes, atomic_write_text, sha256_file, save_gensim_atomic
from src.manifests import TRAINING_COLS, load_manifest, upsert_manifest_row

ROOT = get_project_root()
cfg = yaml.safe_load(open(ROOT / "config/project_config.yaml", encoding="utf-8"))
ver = tuple(int(x) for x in cfg["config_version"].lstrip("v").split("."))
assert ver >= (0, 3, 0), f"need config v0.3.0+, found {cfg['config_version']}"
CFG_SHA = sha256_file(ROOT / "config/project_config.yaml")
E = cfg["embeddings"]
print("config", cfg["config_version"], f"dim={E['dim']} window={E['window']} neg={E['negative']} epochs={E['epochs']} min_count={E['min_count']}")

try: import gensim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "gensim"])
    import gensim
from gensim.models import Word2Vec
print("gensim", gensim.__version__)

import logging
ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOGP = ROOT / f"logs/04_train__{ts}__cfg-{cfg['config_version']}.log"
LOGP.parent.mkdir(parents=True, exist_ok=True)
lg = logging.getLogger("train"); lg.setLevel(logging.INFO); lg.handlers.clear()
fh = logging.FileHandler(LOGP); fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
sh = logging.StreamHandler(sys.stdout); sh.setLevel(logging.WARNING)
lg.addHandler(fh); lg.addHandler(sh)

for d in ["models/word2vec", "models/checkpoints", "diagnostics/model_stability"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

TMP = resolve_tmp(ROOT, cfg)
TMAN = ROOT / "manifests/training_manifest.csv"
if not TMAN.exists():
    atomic_write_text(TMAN, ",".join(TRAINING_COLS) + "\n")

def trows(): return load_manifest(TMAN)
def tupsert(row):
    row = dict(row); row["seed"] = str(row["seed"])
    upsert_manifest_row(TMAN, row, ["model_id", "seed"], TRAINING_COLS)

def atomic_bytes(path: Path, data: bytes): atomic_write_bytes(path, data)
def atomic_text(path: Path, text: str): atomic_write_text(path, text)
def sha_of(p: Path): return sha256_file(p)
today = datetime.datetime.now(datetime.timezone.utc).date().isoformat()
print("setup ready; training rows:", len(trows()), "| tmp:", TMP)

In [ ]:
# Cell 3 — Refresh corpus_master.csv (monthly rollup + periods + shard coverage). The one big metadata CSV.
import glob
PDEF = [r for r in csv.DictReader(open(ROOT / "config/period_definitions.csv", encoding="utf-8")) if not r["model_id"].startswith("#")] if (ROOT / "config/period_definitions.csv").exists() else []
def period_for(sub, mm):
    probe = mm + "-15"
    for r in PDEF:
        if r["subreddit_or_group"] == sub and r["start_date"][:7] <= mm and probe <= r["end_date"]:
            return r["model_id"], r["sufficiency"]
    return "", ""
shard_months = {}
SMAN = ROOT / "manifests/shard_manifest.csv"
if SMAN.exists():
    for r in csv.DictReader(open(SMAN, encoding="utf-8")):
        try:
            for m in {r["min_ts"][:7], r["max_ts"][:7]}:
                shard_months[(r["subreddit"], m)] = shard_months.get((r["subreddit"], m), 0) + 1
        except Exception: pass
ROLL = ROOT / "metadata/monthly_rollup.csv"
out_lines = []
if ROLL.exists():
    for r in csv.DictReader(open(ROLL, encoding="utf-8")):
        pid, suf = period_for(r["subreddit"], r["month"])
        r["period_id"], r["sufficiency"] = pid, suf
        r["shards_present"] = shard_months.get((r["subreddit"], r["month"]), 0)
        r["master_built"], r["master_config"] = today, cfg["config_version"]
        out_lines.append(r)
MASTER = ROOT / "metadata/corpus_master.csv"
if out_lines:
    cols = list(out_lines[0].keys())
    tmp = MASTER.with_suffix(".tmp")
    f = open(tmp, "w", newline="", encoding="utf-8"); w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader(); w.writerows(out_lines); f.flush(); os.fsync(f.fileno()); f.close(); os.replace(tmp, MASTER)
    print(f"corpus_master.csv: {len(out_lines)} rows -> {MASTER}")
else:
    print("no monthly rollup yet — master deferred (run Notebook 02 first; DRY_RUN still works)")


In [ ]:
# Cell 4 — REGISTRY (rewritten): expected = sufficient periods x seeds; COMPLETE requires file+checksum.
wanted = []
for r in PDEF:
    if r["sufficiency"] not in ("sufficient", "axis_grade", "marginal_merge_first"): continue
    if MODEL_FILTER_SUB and r["subreddit_or_group"] != MODEL_FILTER_SUB: continue
    if PERIOD_FILTER and r["model_id"] not in PERIOD_FILTER: continue
    for s in (SEED_FILTER or E["seed_candidates"]):
        wanted.append((r, int(s)))
if MAX_MODELS: wanted = wanted[:MAX_MODELS]
print(f"wanted trainings: {len(wanted)}")
def store_dir(corpus, group): return ROOT / "models/word2vec" / corpus / str(group).lower()
def stem_of(group, period, seed):
    g = "".join(c for c in str(group).lower() if c.isalnum())
    per = period.split("__")[-1] if "__" in period else period
    return f"w2v__{g}__{per}__cfg-{cfg['config_version']}__seed-{seed}"
known = {(r["model_id"], r["seed"]): r for r in trows()}
todo, reg = [], []
for (r, s) in wanted:
    mid = r["model_id"]
    mp = store_dir(r["corpus_type"], r["subreddit_or_group"]) / (stem_of(r["subreddit_or_group"], mid, s) + ".model")
    kr = known.get((mid, str(s)), {})
    ok = kr.get("status") == "complete" and mp.exists() and mp.stat().st_size > 0
    if ok and sha_of(mp) != kr.get("model_sha256", ""): ok = False
    st = "complete" if ok else kr.get("status", "pending")
    if kr.get("status") == "complete" and not ok: st = "validation_failed(file_or_checksum)"
    reg.append((mid, s, st))
    if not ok: todo.append((r, s))
from collections import Counter
print(Counter(s for _, _, s in reg))
for mid, s, st in reg[:20]: print(f"  {st:12s} {mid} seed={s}")
print(f"to train: {len(todo)}")


In [ ]:
# Cell 5 — STREAMING CORPUS + TRAIN LOOP. One model in RAM; per-epoch checkpoints; sidecar provenance.
class ShardSentences:
    'Re-iterable stream of token lists over validated .jsonl.gz shards. Never a list.'
    def __init__(self, paths): self.paths = list(paths)
    def __iter__(self):
        for p in self.paths:
            with gzip.open(p, "rt", encoding="utf-8") as f:
                for line in f:
                    if not line or line[0] == "#": continue
                    try: rec = json.loads(line)
                    except Exception: continue
                    toks = rec.get("tokens")
                    if isinstance(toks, list) and len(toks) >= 3: yield toks

def shards_for(sub, pid):
    'Exact shard lookup via shard_manifest.csv. No silent fallback across multi-year corpora.'
    sman_path = ROOT / "manifests/shard_manifest.csv"
    if sman_path.exists():
        rows = [r for r in csv.DictReader(open(sman_path, encoding="utf-8")) if r.get("status") == "complete"]
        matched = [r["path"] for r in rows if r.get("period_id") == pid and (r.get("subreddit") == sub or (sub == "REF_POOLED" and r.get("subreddit") == "REF"))]
        existing = [Path(p) for p in matched if Path(p).exists()]
        if existing: return sorted(existing)
        
    base = ROOT / "shards/tokenized"
    slug = "ref" if sub == "REF_POOLED" else "".join(c for c in str(sub).lower() if c.isalnum())
    span = pid.split("__")[-1]
    return sorted(base.rglob(f"*{slug}*{span}*.jsonl.gz"))

def spec_hash(d): return hashlib.sha256(json.dumps(d, sort_keys=True).encode()).hexdigest()[:12]

def train_one(r, seed):
    mid, grp, corpus = r["model_id"], r["subreddit_or_group"], r["corpus_type"]
    sdir = store_dir(corpus, grp); sdir.mkdir(parents=True, exist_ok=True)
    stem = stem_of(grp, mid, seed)
    mpath = sdir / (stem + ".model")
    
    spec = {"dim": E["dim"], "window": E["window"], "sg": 1, "negative": E["negative"],
            "epochs": E["epochs"], "min_count": E["min_count"], "sample": E["subsample"]}
    row = {"model_id": mid, "corpus_type": corpus, "subreddit_or_group": grp, "period_id": mid,
           "spec_hash": spec_hash(spec), "dim": E["dim"], "window": E["window"], "sg": 1,
           "negative": E["negative"], "epochs": E["epochs"], "min_count": E["min_count"],
           "max_vocab": E["max_vocab"], "subsample": E["subsample"], "workers": E["workers"],
           "lr": str(E["lr"]), "seed": seed, "vocab_size": "", "words_processed": "",
           "epochs_done": 0, "train_secs": "", "peak_ram_mb": "", "model_path": str(mpath),
           "vectors_path": "", "model_sha256": "", "vectors_sha256": "",
           "config_version": cfg["config_version"], "status": "in_progress", "diagnostics_path": ""}
    tupsert(row)
    
    try:
        paths = shards_for(grp, mid)
        assert paths, f"no shards found for {grp}/{mid} — run Notebook 03 first"
        t0 = time.time()
        
        initial_lr = float(E["lr"]["initial"]) if isinstance(E["lr"], dict) else float(E.get("lr", 0.025))
        min_lr = float(E["lr"]["min"]) if isinstance(E["lr"], dict) else 0.0001
        total_epochs = int(E["epochs"])

        model = Word2Vec(vector_size=E["dim"], window=E["window"], sg=1, negative=E["negative"],
                         min_count=E["min_count"], sample=E["subsample"], workers=E["workers"],
                         seed=int(seed), alpha=initial_lr, min_alpha=min_lr, epochs=1)
        model.build_vocab(ShardSentences(paths))
        
        for ep in range(total_epochs):
            # Preserved linear learning rate decay schedule across epochs
            ep_alpha = initial_lr - (initial_lr - min_lr) * (ep / total_epochs)
            ep_min_alpha = initial_lr - (initial_lr - min_lr) * ((ep + 1) / total_epochs)
            model.train(ShardSentences(paths), total_examples=model.corpus_count, epochs=1,
                        start_alpha=ep_alpha, end_alpha=ep_min_alpha)
            ck = ROOT / "models/checkpoints" / (stem + f".ep{ep + 1}.model")
            save_gensim_atomic(model, ck)
            row["epochs_done"] = ep + 1; tupsert(row)
            
        save_gensim_atomic(model, mpath)
        assert mpath.stat().st_size > 0
        m2 = Word2Vec.load(str(mpath)); assert len(m2.wv) == len(model.wv); del m2
        try:
            import resource; peak = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1)
        except Exception: peak = "n/a"
        nfo = {"model_id": mid, "seed": seed, "spec": spec, "config_version": cfg["config_version"],
               "config_sha256": CFG_SHA, "gensim": gensim.__version__, "sys": sys.version.split()[0],
               "shards": [str(p) for p in paths], "shard_sha256": {str(p): sha_of(p) for p in paths},
               "vocab_size": len(model.wv), "corpus_count": model.corpus_count,
               "train_secs": round(time.time() - t0, 1), "period_row": r}
        atomic_bytes(sdir / (stem + ".nfo.json"), json.dumps(nfo, indent=2).encode())
        row.update({"status": "complete", "vocab_size": len(model.wv), "words_processed": model.corpus_total_words,
            "train_secs": nfo["train_secs"], "peak_ram_mb": peak, "model_sha256": sha_of(mpath)}); tupsert(row)
        print(f"OK {stem}: vocab={len(model.wv)} words={model.corpus_total_words} secs={nfo['train_secs']}")
        del model; gc.collect()
        return True
    except Exception as e:
        lg.exception(mid)
        row.update({"status": "failed"}); tupsert(row)
        with open(ROOT / "manifests/failure_log.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps({"ts": datetime.datetime.now(datetime.timezone.utc).isoformat(), "stage": "train",
                "unit": mid, "seed": seed, "error": str(e)[:300], "config_version": cfg["config_version"]}) + "\n")
        print(f"FAIL {mid} seed={seed}: {str(e)[:140]}"); return False

print("train fn ready (safe atomic multi-file save + linear LR schedule)")

In [ ]:
# Cell 6 — EXECUTE (or DRY_RUN synthetic proof in tmp only; never touches the model store).
if DRY_RUN:
    import random
    random.seed(7); vocab = [f"w{i}" for i in range(60)] + ["not", "never", "good", "bad"]
    dp = TMP / "dry_shards"; dp.mkdir(parents=True, exist_ok=True)
    sp = dp / "dry__0000.jsonl.gz"
    f = gzip.open(sp, "wt", encoding="utf-8"); f.write('#manifest {"dry": true}\n')
    for i in range(600):
        f.write(json.dumps({"tokens": [random.choice(vocab) for _ in range(random.randint(5, 25))]}) + "\n")
    f.close()
    m = Word2Vec(vector_size=10, window=3, sg=1, negative=5, min_count=2, workers=1, seed=7, epochs=1)
    m.build_vocab(ShardSentences([sp])); m.train(ShardSentences([sp]), total_examples=m.corpus_count, epochs=1)
    assert len(m.wv) > 10 and "not" in m.wv, "dry-run model invalid"
    print(f"DRY_RUN PASS: vocab={len(m.wv)} negation_kept={'not' in m.wv}; store untouched")
    ok = fail = 0
else:
    if not PDEF: raise SystemExit("STOP: period_definitions.csv is empty — run Notebook 02 with FREEZE_PERIODS=True (no silent training without frozen periods).")
    ok = fail = 0
    for r, s in todo:
        if train_one(r, s): ok += 1
        else: fail += 1
    print(f"trained ok={ok} failed={fail}")


In [ ]:
# Cell 7 — VALIDATE new models (reopen; data-driven probes; negation check) + SEED EVALUATION (Stage 9).
# Rule (prespecified, config seed_eval): production seed = max mean cross-seed neighbor-Jaccard
# over top-N frequent words; ties -> lowest seed id. NEVER most-favorable-substantive. Flags, not deletions.
import numpy as np
fresh = {}
for (mid, s, st) in reg:
    kr = {(r["model_id"], r["seed"]): r for r in trows()}.get((mid, str(s)))
    if not kr or kr["status"] != "complete" or st == "complete": continue
    try:
        m = Word2Vec.load(kr["model_path"])
        counts = sorted(((w, m.wv.get_vecattr(w, "count")) for w in m.wv.index_to_key), key=lambda x: -x[1])
        probes = [w for w, _ in counts[:8]]
        dg = {"model_id": mid, "seed": s, "vocab": len(m.wv), "top": probes, "negation_kept": "not" in m.wv,
              "neighbors": {w: [x for x, _ in m.wv.most_similar(w, topn=5)] for w in probes[:3]}}
        dp = ROOT / "diagnostics/model_stability" / (Path(kr["model_path"]).stem + ".json")
        atomic_bytes(dp, json.dumps(dg, indent=2).encode())
        r2 = dict(kr); r2["diagnostics_path"] = str(dp); tupsert(r2)
        fresh[(mid, s)] = m
        print(f"VALID {Path(kr['model_path']).stem}: vocab={len(m.wv)} negation={dg['negation_kept']}")
    except Exception as e:
        lg.error(f"validate {mid}: {e}"); print(f"VALIDATE-FAIL {mid}: {str(e)[:120]}")
SE = cfg.get("seed_eval", {})
DN, DK = int(SE.get("diag_words", 200)), int(SE.get("neighbor_k", 10))
MINJ = float(SE.get("min_cross_seed_jaccard", 0.30))
from collections import defaultdict as _dd
by_model = _dd(list)
for r in trows():
    if r["status"] == "complete": by_model[r["model_id"]].append(r)
prod, seedrep = {}, []
for mid, rows in sorted(by_model.items()):
    if PERIOD_FILTER and mid not in PERIOD_FILTER: continue
    if MODEL_FILTER_SUB and rows[0]["subreddit_or_group"] != MODEL_FILTER_SUB: continue
    mods = {}
    for r in rows:
        try: mods[int(r["seed"])] = Word2Vec.load(r["model_path"])
        except Exception as e: lg.error(f"seedload {mid}: {e}")
    if len(mods) < 2:
        seedrep.append({"model_id": mid, "note": "single-seed — no band; train more seeds", "production_seed": sorted(mods)[0] if mods else None})
        if mods: prod[mid] = sorted(mods)[0]
        continue
    common = set.intersection(*[set(m.wv.index_to_key[:5000]) for m in mods.values()])
    freq = sorted(common, key=lambda w: -min(m.wv.get_vecattr(w, "count") for m in mods.values()))[:DN]
    nb = {s: {w: set(x for x, _ in m.wv.most_similar(w, topn=DK)) for w in freq if w in m.wv} for s, m in mods.items()}
    def jac(a, b): return len(a & b) / len(a | b) if (a | b) else 0.0
    meanj = {}
    for s in mods:
        js = [jac(nb[s][w], nb[o][w]) for o in mods if o != s for w in freq if w in nb[s] and w in nb[o]]
        meanj[s] = sum(js) / max(1, len(js))
    best = sorted(meanj, key=lambda s: (-meanj[s], s))[0]
    prod[mid] = best
    flag = "OK" if meanj[best] >= MINJ else f"FLAG below_min_jaccard({MINJ}) — investigate, do not silently drop"
    seedrep.append({"model_id": mid, "mean_cross_seed_jaccard": {str(k): round(v, 3) for k, v in meanj.items()},
                    "production_seed": best, "status": flag})
    print(f"SEED {mid}: production={best} jaccard={ {k: round(v,3) for k,v in meanj.items()} } {flag}")
    for m in mods.values(): del m
    gc.collect()
atomic_text(ROOT / "models/production_seed.json", json.dumps({"built": today, "config": cfg["config_version"], "seeds": prod, "report": seedrep}, indent=2))
print(f"production seeds -> models/production_seed.json ({len(prod)} models)")


In [ ]:
# Cell 8 — REGISTRY REPRINT + END-OF-RUN SUMMARY.
from collections import Counter
cur = {(r["model_id"], r["seed"]): r["status"] for r in trows()}
print(dict(Counter(cur.get((r["model_id"], str(s)), "pending") for (r, s) in wanted)))
print("=" * 70)
print(f"TRAINING COMPLETE ok={ok} failed={fail} | production seeds chosen for {len(prod)} models")
print("store: models/word2vec/<corpus>/<group>/w2v__*.model + .nfo.json | registry: manifests/training_manifest.csv")
print("rerun safe: YES (checksum-verified skips; failed-only retries; checkpoints per epoch)")
print("next: 05_vectors_inspect.ipynb")
print("=" * 70)
